## Introduction

After compressing a model (pruning, quantization, sparsification), you want to know: *how much did it actually improve?* fasterbench's `ComparisonReport` answers this by benchmarking both versions and generating a professional report with metric deltas, improvement rankings, and optional radar charts.

This tutorial shows the full workflow: benchmark → compress → benchmark again → generate report.

## Setup

In [ ]:
import torch, torch.nn as nn
from fasterbench import benchmark, Report, ComparisonReport

## Step 1: Benchmark the Original Model

In [ ]:
from torchvision.models import resnet18

model = resnet18(pretrained=True)
x = torch.randn(1, 3, 224, 224)

result_before = benchmark(model, x, metrics=["size", "speed", "compute"])
result_before.summary()

/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


═══ Size ════════════════════════════════════
  Disk:   44.67 MiB
  Params: 11.69M
═══ Speed ═══════════════════════════════════
  cpu: 27.41 ms  │  36.5 inf/s  │  p99: 28.39 ms
  cuda: 0.65 ms  │  1527.9 inf/s  │  p99: 0.70 ms
═══ Compute ═════════════════════════════════
  MACs: 1824.0 M


## Step 2: Compress the Model

Use any compression technique — pruning, quantization, sparsification:

In [ ]:
# Example: structured pruning with fasterai
from fasterai.prune.pruner import Pruner
from fasterai.core.criteria import large_final

pruner = Pruner(model, pruning_ratio=0.5, context='local', criteria=large_final,
                example_inputs=x)
pruner.prune_model()

Ignoring output layer: fc
Total ignored layers: 1


RuntimeError: Inference tensors cannot be saved for backward. Please do not use Tensors created in inference mode in computation tracked by autograd. To work around this, you can make a clone to get a normal tensor and use it in autograd, or use `torch.no_grad()` instead of `torch.inference_mode()`.

## Step 3: Benchmark the Compressed Model

In [ ]:
result_after = benchmark(model, x, metrics=["size", "speed", "compute"])
result_after.summary()

## Step 4: Generate Comparison Report

In [ ]:
report = ComparisonReport(
    result_before, result_after,
    before_name="ResNet-18 (original)",
    after_name="ResNet-18 (50% pruned)",
    title="Pruning Compression Report"
)

report.summary()

## Export to HTML or Markdown

Generate shareable reports:

In [ ]:
# HTML report with embedded radar chart
report.to_html("compression_report.html", include_charts=True)

# Markdown report (great for GitHub PRs)
md = report.to_markdown("compression_report.md")
print(md[:200])

## Single Model Report

`Report` generates a standalone report for a single model:

In [ ]:
single = Report(result_before, model_name="ResNet-18", 
               description="Baseline ImageNet classifier")
single.summary()

# Also supports HTML and Markdown export
single.to_html("resnet18_report.html")

## Programmatic Access

Access report data as dictionaries for further processing:

In [ ]:
# Get all deltas as a list
for d in report.deltas:
    if d.improved:
        print(f"{d.label}: {d.delta_pct:+.1f}% ({'improved' if d.improved else 'regressed'})")

# Top 3 improvements
for d in report.top_improvements(3):
    print(f"  {d.label}: {d.before:.0f} → {d.after:.0f}")

# Serialize to dict (for JSON, databases, etc.)
data = report.as_dict()

---

## Summary

| Tool / Function | Purpose |
|----------------|----------|
| `Report(result)` | Single model report |
| `ComparisonReport(before, after)` | Before/after comparison |
| `.summary()` | Console output |
| `.to_html(path)` | HTML with radar charts |
| `.to_markdown(path)` | Markdown (great for PRs) |
| `.deltas` | List of `ReportMetricDelta` |
| `.top_improvements(n)` | Top N improvements sorted by impact |

---

## See Also

- [Benchmark](../analysis/benchmark.html) — Unified benchmarking API
- [Visualization](../visualization/plot.html) — Radar plots for visual comparison
- [Report API](../analysis/report.html) — Full API reference